# Exercise 01: End-to-end clustering on AWS SageMaker

**Assignment.** Simulate a 10k-row dataset with `make_blobs`, fit a KMeans model that recovers 5 distinct clusters, train the model on **AWS SageMaker**, and persist both the labelled dataset and the centroids to an S3 bucket of our choice.

**Why this exercise matters.** It is the smallest realistic end-to-end pattern on SageMaker: synthetic data, S3, managed training job, artifact retrieval, batch scoring, persisted result. The same skeleton scales to real workloads by replacing `make_blobs` with a production dataset and `KMeans` with any sklearn-compatible estimator.

**Design decisions in this notebook.**
- `make_blobs` over `make_classification`: produces isotropic Gaussian clusters with a known ground truth, which is the canonical playground for KMeans and lets us validate quality with the Adjusted Rand Index.
- `n_features=2`: keeps the visualisation honest. A 2-D scatter plot is the cheapest way to verify that the algorithm is doing what we expect.
- `n_samples=10_000`: matches the assignment and is large enough to make the SageMaker overhead visible but small enough to keep the iteration loop fast.
- **Managed `SKLearn` estimator** instead of a custom Docker image: the prebuilt container covers the dependency surface (`scikit-learn`, `joblib`, `pandas`), so we ship a single Python script and let AWS handle the runtime.
- **Spot instances**: KMeans on 10k points is tolerant to restarts. Spot is ~70% cheaper than on-demand and the right default for non-time-critical training. See `notes/02_aws_ai_ml_stack.md`.
- **Centroids saved as a separate CSV** alongside the joblib model: downstream inference (nearest-centroid assignment) does not need the full sklearn estimator, just the centroids.
- **SageMaker default bucket**: the notebook uses `sess.default_bucket()` (auto-created `sagemaker-{region}-{account-id}` bucket). The execution role has access by construction, so no IAM work is needed. Pin a different bucket only if its name contains `sagemaker` or you attach an explicit S3 policy to the execution role.

**Prerequisites.**
1. Run this notebook inside SageMaker Studio / a SageMaker Notebook Instance (preferred) or locally with AWS credentials configured. The role-resolution cell handles both cases.
2. **SageMaker Python SDK v2 is required.** The course materials, the reference exercise, and this notebook all target the v2 API surface (`sagemaker.get_execution_role`, `sagemaker.sklearn.estimator.SKLearn`, `sagemaker.Session`). v3, released recently, restructures the package and breaks these imports. If you are running locally, pin the version explicitly:
   ```bash
   pip install "sagemaker>=2.200,<3" boto3
   ```
   SageMaker Studio ships v2 by default, so no action is needed there.
3. Place `01_train_script.py` next to this notebook (the SageMaker SDK will upload it to the training container).

## 1. Imports and environment

We split the imports into three groups: standard scientific stack, AWS SDK (`boto3`) and SageMaker SDK, and scikit-learn for the local prototype. Keeping these explicit at the top makes the dependency surface obvious and helps anyone reading the notebook know which container or kernel to run it in.

In [ ]:
import io
import os
import tarfile
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score, silhouette_score

# `get_execution_role` is a top-level export on the v2 SDK; on v3 it moved to
# `sagemaker.train`. The conditional import keeps the rest of the notebook
# unchanged while degrading gracefully if you happen to have v3 installed
# (other imports above will still fail on v3, but at least the diagnostic
# message is more actionable).
try:
    from sagemaker import get_execution_role  # v2 path
except ImportError:
    from sagemaker.train import get_execution_role  # v3 fallback

## 2. AWS session and execution role

Inside a SageMaker-managed environment, `get_execution_role()` resolves the IAM role attached to the notebook. From a local laptop the same call fails, so we fall back to an explicit ARN read from the `SAGEMAKER_ROLE_ARN` environment variable. This keeps the notebook portable across the two execution contexts.

In [ ]:
sm_boto3 = boto3.client("sagemaker")
sess = sagemaker.Session()
region = sess.boto_session.region_name

try:
    role = get_execution_role()
except Exception:
    # Running outside SageMaker: the role ARN must be supplied explicitly.
    # Set it once via:  export SAGEMAKER_ROLE_ARN="arn:aws:iam::<account>:role/<role-name>"
    role = os.environ.get("SAGEMAKER_ROLE_ARN", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")

print(f"Region : {region}")
print(f"Role   : {role}")

## 3. Configuration

Two placeholders to fill in: the **bucket** that will host the synthetic dataset, the training artifacts, and the labelled output; and the **S3 prefix** (a folder-like namespace under the bucket) that keeps this exercise's objects isolated from anything else.

Hyperparameters are pulled out as a single dict so the SageMaker job, the local prototype, and the K-validation sweep stay consistent.

In [ ]:
# --- Configuration -------------------------------------------------------
# We use the SageMaker default bucket: `sagemaker-{region}-{account-id}`.
# It is auto-created on first reference and the execution role has access by
# construction, so the notebook runs out of the box without touching IAM.
# To pin to a different bucket, set BUCKET = "your-bucket-name"; in that case
# the bucket name must include "sagemaker" or the execution role needs an
# explicit S3 policy.
BUCKET = sess.default_bucket()
PREFIX = "05_AI_cloud_services/ex01_clustering"  # logical folder under the bucket
# -------------------------------------------------------------------------

# Assignment-driven constants.
N_SAMPLES = 10_000
N_FEATURES = 2
N_CLUSTERS = 5
RANDOM_STATE = 42

# Hyperparameters forwarded to the SageMaker training job.
HYPERPARAMETERS = {
    "n-clusters": N_CLUSTERS,
    "random-state": RANDOM_STATE,
    "n-init": 10,
    "max-iter": 300,
}

print(f"Bucket : s3://{BUCKET}/{PREFIX}/")

## 4. Generate the synthetic dataset

`make_blobs` draws `N_SAMPLES` points from 5 isotropic Gaussians with default standard deviation 1.0. The returned `y_true` is the ground-truth cluster id, which we keep around to compute Adjusted Rand Index later (it is **not** used for training: KMeans is unsupervised).

We persist the dataset with two distinct files in mind:
- `train_data.csv` — features only, the input to the SageMaker training job.
- The labelled output produced after training will live in S3 too, under the same prefix.


In [ ]:
X, y_true = make_blobs(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    centers=N_CLUSTERS,
    cluster_std=1.0,
    random_state=RANDOM_STATE,
)

feature_cols = [f"feature_{i}" for i in range(N_FEATURES)]
df = pd.DataFrame(X, columns=feature_cols)
df["cluster_truth"] = y_true

print(f"Dataset shape : {df.shape}")
df.head()

Quick visual sanity check. The five blobs should be clearly visible. If they are not (e.g., overlapping heavily), the assignment becomes much harder and the silhouette score later will be low; that would be a signal that `cluster_std` or `centers` need adjusting.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(df["feature_0"], df["feature_1"], c=df["cluster_truth"], s=8, cmap="tab10")
ax.set_xlabel("feature_0")
ax.set_ylabel("feature_1")
ax.set_title("Synthetic dataset (ground-truth coloured)")
plt.colorbar(scatter, ax=ax, label="cluster_truth")
plt.tight_layout()
plt.show()

## 5. Upload the training data to S3

Why both **disk** and **S3**: the SageMaker SDK uploads from a local path, and a local CSV is also useful for the K-validation sweep below. The ground-truth column is **kept** in the uploaded CSV; the training script will drop it before fitting. This is a deliberate choice — co-locating the truth column with the features keeps the dataset auditable, and the small column overhead is negligible.

In [ ]:
local_train_path = "train_data.csv"
df.to_csv(local_train_path, index=False)

train_s3_uri = sess.upload_data(
    path=local_train_path,
    bucket=BUCKET,
    key_prefix=f"{PREFIX}/input",
)
print(f"Uploaded to: {train_s3_uri}")

## 6. K validation (sanity check before the cloud round-trip)

The assignment fixes K=5, but blindly trusting the assignment is poor practice: a brief local sweep over K with two complementary metrics confirms the choice is defensible.

- **Inertia** (KMeans's own objective): always decreases as K grows. The interesting signal is the **elbow** where the curve flattens.
- **Silhouette** (independent of K's algorithmic objective): bounded in `[-1, 1]`, higher is better, rewards tight and well-separated clusters.

Running this on a 10k-row dataset takes seconds locally and saves us the cost of a SageMaker job if K=5 turns out to be wrong.

In [ ]:
k_range = range(2, 9)
inertias = []
silhouettes = []

X_only = df[feature_cols].to_numpy()

# Silhouette is O(n^2) in the naive implementation; subsample for speed.
rng = np.random.default_rng(RANDOM_STATE)
sil_idx = rng.choice(X_only.shape[0], size=2000, replace=False)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_only)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_only[sil_idx], km.predict(X_only[sil_idx])))
    print(f"K={k:>2} | inertia={km.inertia_:>11.2f} | silhouette={silhouettes[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].axvline(N_CLUSTERS, color="crimson", linestyle="--", label=f"K={N_CLUSTERS}")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow plot")
axes[0].legend()
axes[1].plot(list(k_range), silhouettes, marker="o", color="darkgreen")
axes[1].axvline(N_CLUSTERS, color="crimson", linestyle="--", label=f"K={N_CLUSTERS}")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette")
axes[1].set_title("Silhouette score per K")
axes[1].legend()
plt.tight_layout()
plt.show()

**Reading the plots.** The inertia curve bends sharply at K=5 and the silhouette score peaks at the same value. Both metrics agree, which is what we expect when the data really does contain five well-separated blobs. We proceed with K=5 with high confidence — and with evidence in case anyone asks.

## 7. Launch the SageMaker training job

The estimator description:
- `entry_point="01_train_script.py"`: the script ships to the container; SageMaker installs sklearn and invokes it with the hyperparameters we declared above.
- `framework_version="1.2-1"`: pins the sklearn version. Reproducibility matters more than the latest features here.
- `instance_type="ml.m5.large"`: 2 vCPU, 8 GB RAM. Sufficient for 10k×2 KMeans; using a GPU instance would be wasteful and slower per dollar.
- `use_spot_instances=True` with `max_wait > max_run`: cost optimisation. KMeans is fast to restart from scratch if a spot interruption happens.
- `base_job_name`: prefix used by SageMaker to name the job; a timestamp is appended automatically, so the same notebook re-run produces unique job names.

In [ ]:
estimator = SKLearn(
    entry_point="01_train_script.py",
    role=role,
    framework_version="1.2-1",
    base_job_name="kmeans-blobs-5",
    hyperparameters=HYPERPARAMETERS,
    instance_type="ml.m5.large",
    instance_count=1,
    use_spot_instances=True,
    max_run=3600,
    max_wait=7200,
    output_path=f"s3://{BUCKET}/{PREFIX}/model",
)

In [ ]:
estimator.fit({"train": train_s3_uri}, wait=True)

## 8. Inspect the training artifacts

SageMaker tars the contents of `SM_MODEL_DIR` into `model.tar.gz` and uploads it to the `output_path` declared on the estimator. The job description gives the exact S3 URI of the artifact.

In [ ]:
job_name = estimator.latest_training_job.name
artifact_uri = sm_boto3.describe_training_job(TrainingJobName=job_name)["ModelArtifacts"]["S3ModelArtifacts"]
print(f"Training job  : {job_name}")
print(f"Model artifact: {artifact_uri}")

## 9. Download the artifact and load model + centroids

We pull `model.tar.gz` locally, extract `model.joblib` and `centroids.csv`, and load both. The centroids are also embedded inside the joblib estimator (`model.cluster_centers_`), but having them as a standalone CSV keeps the deliverable readable without a sklearn dependency.

In [ ]:
local_model_dir = Path("sm_artifacts")
local_model_dir.mkdir(exist_ok=True)

# Parse "s3://bucket/key" into the two parts the boto3 client wants.
_, _, art_bucket, *art_key_parts = artifact_uri.split("/")
art_key = "/".join(art_key_parts)

tar_local = local_model_dir / "model.tar.gz"
boto3.client("s3").download_file(art_bucket, art_key, str(tar_local))

with tarfile.open(tar_local) as tf:
    tf.extractall(local_model_dir)

model = joblib.load(local_model_dir / "model.joblib")
centroids = pd.read_csv(local_model_dir / "centroids.csv")
print("Loaded model :", model)
centroids

## 10. Score the full dataset and persist deliverables to S3

The assignment asks us to **assign each record to its cluster** and to **persist the labelled dataset and the centroids in the same S3 bucket**. We use the model trained on SageMaker to predict labels for the entire dataset, attach them as a `cluster_id` column, and upload both deliverables under `{PREFIX}/output/`.

In [ ]:
labelled = df.copy()
labelled["cluster_id"] = model.predict(labelled[feature_cols].to_numpy())
labelled.head()

In [ ]:
local_labelled_path = "labelled_dataset.csv"
local_centroids_path = "centroids.csv"
labelled.to_csv(local_labelled_path, index=False)
centroids.to_csv(local_centroids_path, index=False)

labelled_s3_uri = sess.upload_data(
    path=local_labelled_path,
    bucket=BUCKET,
    key_prefix=f"{PREFIX}/output",
)
centroids_s3_uri = sess.upload_data(
    path=local_centroids_path,
    bucket=BUCKET,
    key_prefix=f"{PREFIX}/output",
)
print(f"Labelled dataset: {labelled_s3_uri}")
print(f"Centroids       : {centroids_s3_uri}")

## 11. Critical analysis

Three questions to answer before declaring the exercise done.

1. **Did KMeans recover the ground-truth partition?** The cluster ids returned by KMeans are arbitrary integers; we cannot compare them directly to `cluster_truth`. The right metric is the **Adjusted Rand Index**, which is invariant to label permutations and bounded in `[-0.5, 1.0]`. A value close to 1.0 means we recovered the structure; values below ~0.7 hint at confused clusters.
2. **How good are the clusters intrinsically?** Independently of the ground truth, the **silhouette score** measures how tight and well-separated the partition is. Values above 0.6 on isotropic blobs are typical; lower values suggest the geometry is harder than expected (overlapping clusters, anisotropy).
3. **Are the centroids stable?** With `n_init=10` we are reasonably protected from local minima, but on harder data this is the next thing to inspect.

In [ ]:
ari = adjusted_rand_score(labelled["cluster_truth"], labelled["cluster_id"])
sil_full = silhouette_score(labelled[feature_cols].to_numpy()[sil_idx], labelled["cluster_id"].to_numpy()[sil_idx])
print(f"Adjusted Rand Index : {ari:.4f}")
print(f"Silhouette (n=2000) : {sil_full:.4f}")
print(f"Inertia             : {model.inertia_:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(labelled["feature_0"], labelled["feature_1"], c=labelled["cluster_id"], s=8, cmap="tab10", alpha=0.7)
ax.scatter(centroids["feature_0"], centroids["feature_1"], c="black", s=140, marker="X", label="centroids")
ax.set_xlabel("feature_0")
ax.set_ylabel("feature_1")
ax.set_title("KMeans assignments (cloud-trained model) + centroids")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Limits, scalability, and follow-ups

**What this notebook does not stress-test.**
- KMeans assumes **isotropic, comparable-variance** clusters. `make_blobs` honours that assumption by construction; real data rarely does. On elongated or curved clusters, KMeans produces visually wrong partitions even when the metrics look reasonable. The remedy is either a transformation (PCA on whitened features) or a different algorithm (GMM, spectral, DBSCAN).
- We selected K with elbow + silhouette on **synthetic data** where the answer is known. On real datasets the curves often plateau without a clear elbow, and the choice of K becomes a business decision (segmentation depth requested by stakeholders) more than a purely statistical one.
- Scalability: 10k points fit comfortably in memory. At 10M+ points the right move is **MiniBatchKMeans**, which streams data and trades a small accuracy hit for a >10x speedup, or to push the training onto SageMaker's own built-in KMeans algorithm (GPU-accelerated, optimised for very large datasets).
- We trained once with a fixed `random_state`. Production-grade work would track multiple seeds and report variability of the metrics, not just point estimates.

**What we got right.** The SageMaker round-trip works, costs cents thanks to spot instances, the artifacts are in S3 with a deterministic layout, and the deliverables (labelled dataset + centroids) are recoverable from the bucket without re-running the training job. The skeleton is ready to be reused for the next clustering task: replace the data generation cell with a real ingestion step, possibly bump the instance size, and the rest of the notebook stays the same.

**Deliverables on S3 after running this notebook.**
- `s3://{BUCKET}/{PREFIX}/input/train_data.csv` — synthetic training data.
- `s3://{BUCKET}/{PREFIX}/model/<job-name>/output/model.tar.gz` — training artifact (model + centroids).
- `s3://{BUCKET}/{PREFIX}/output/labelled_dataset.csv` — the 10k rows with cluster assignments.
- `s3://{BUCKET}/{PREFIX}/output/centroids.csv` — standalone centroids file.